# Realistic Stress-Testing: Complex (Nvidia-like / Apple-like) Networks

This notebook mirrors `03_stress_testing (large network)`'s Ray-based distributed pattern, but runs the named disruption-scenario library from `scripts/disruption_scenarios.py` against the two ~2,300-node complex networks generated in `05_realistic_operational_data` — one anchored on real, publicly-known companies in Nvidia's AI/GPU chip supply chain, one on Apple's consumer-electronics supply chain (see the disclaimer in `05` and in the README: company names are real and cited, all attached numeric figures are synthetic/illustrative).

As in `03`, we distribute thousands of single-node disruption scenarios (plus a handful of real-world-event-inspired regional/material scenarios) across a Ray cluster, for both companies in one combined sweep.

## Cluster Configuration
This notebook was tested on the following Databricks cluster configuration:
- **Databricks Runtime Version:** 17.3 LTS ML (includes Apache Spark 4.0.0, Scala 2.13)
- **Driver Type**
    - Azure: Standard_DS4_v2 (28 GB Memory, 8 Cores)
    - AWS: m5d.2xlarge (32 GB Memory, 8 Cores)
- **Worker Type**
    - Azure: Standard_E4d_v4 (32 GB Memory, 4 Cores)
    - AWS: rd-fleet.xlarge (32 GB Memory, 4 Cores)
- **Number of Workers:** 4
- **Photon Acceleration:** Disabled (Photon boosts Apache Spark workloads; not all ML workloads will see an improvement)
> **Note:** Performance may vary depending on the cluster size, node types, and workload characteristics.

In [0]:
%pip install -r ./requirements.txt --quiet
dbutils.library.restartPython()

In [0]:
import os
import json
import random
import numpy as np
import pandas as pd
import scripts.disruption_scenarios as ds_lib

In [ ]:
catalog = "supply_chain_stress_test"  # Change here
read_schema = "data"                  # Change here
volume = "operational"                # Change here
write_schema = "results"              # Change here

# Make sure that the catalog and the schema exist
_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{write_schema}")

dbutils.widgets.dropdown("run_planet_scale", "yes", ["yes", "no"], "Run planet-scale sweep")
run_planet_scale = dbutils.widgets.get("run_planet_scale") == "yes"

## Get Datasets

Both complex datasets generated in `05_realistic_operational_data`.

In [0]:
complex_datasets = {}
for company in ("nvidia", "apple"):
    with open(f"/Volumes/{catalog}/{read_schema}/{volume}/dataset_realistic_{company}.json", "r") as f:
        complex_datasets[company] = json.load(f)

## Retrieve Databricks Cluster Information
Same helper as `03_stress_testing (large network)`.

In [ ]:
# Databricks-only: get cluster context and min/max nodes
def get_min_max_nodes():
    try:
        import requests
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        host_name = ctx.browserHostName().get()
        host_token = ctx.apiToken().get()
        cluster_id = ctx.clusterId().get()
        response = requests.get(
            f'https://{host_name}/api/2.1/clusters/get?cluster_id={cluster_id}',
            headers={'Authorization': f'Bearer {host_token}'}
        ).json()
        if "autoscale" in response:
            return response['autoscale']["min_workers"], response['autoscale']["max_workers"]
        return 1, response['num_workers']  # fixed-size cluster
    except Exception as e:
        print(f"Warning: Could not fetch min/max nodes from Databricks context: {e}")
        return 1, 4  # fallback default (matches documented 1-4 worker cluster)

min_node, max_node = get_min_max_nodes()

## Ray Cluster Initialization

In [0]:
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

restart = True
if restart is True:
    try:
        shutdown_ray_cluster()
    except Exception:
        pass
    try:
        ray.shutdown()
    except Exception:
        pass

num_cpu_cores_per_worker = 4
num_cpus_head_node = 4

try:
    from mlflow.utils.databricks_utils import get_databricks_env_vars
    mlflow_dbrx_creds = get_databricks_env_vars("databricks")
    os.environ["DATABRICKS_HOST"] = mlflow_dbrx_creds['DATABRICKS_HOST']
    os.environ["DATABRICKS_TOKEN"] = mlflow_dbrx_creds['DATABRICKS_TOKEN']
except Exception as e:
    print(f"Warning: Could not set Databricks env vars: {e}")

ray_conf = setup_ray_cluster(
    min_worker_nodes=min_node,
    max_worker_nodes=max_node,
    num_cpus_head_node=num_cpus_head_node,
    num_cpus_per_node=num_cpu_cores_per_worker,
    num_gpus_head_node=0,
    num_gpus_worker_node=0
)
os.environ['RAY_ADDRESS'] = ray_conf[0]

## Prepare Scenario Data for Distributed Computation

We build the scenario library once per company (single-supplier failures with a 5-60 day recovery-time range, plus the curated real-world regional/material scenarios that apply to that company's dataset), tag each row with which company it belongs to, and combine both into one Ray Dataset.

In [0]:
def build_scenario_rows(company, dataset):
    rng = random.Random(1)
    scenarios = ds_lib.single_supplier_failure_scenarios(dataset, rng, ttr_lo=5, ttr_hi=60)
    scenarios += ds_lib.named_real_world_scenarios(dataset)
    return [
        {
            "company": company,
            "scenario_id": s.scenario_id,
            "scenario_name": s.name,
            "scenario_type": s.scenario_type,
            "real_world_basis": s.real_world_basis,
            "disrupted_nodes": list(s.disrupted_nodes),
            "ttr": s.ttr,
        }
        for s in scenarios
    ]

rows = []
for company, dataset in complex_datasets.items():
    rows.extend(build_scenario_rows(company, dataset))

df = pd.DataFrame(rows)
print(f"{len(df)} total scenarios across {df['company'].nunique()} companies")
df = ray.data.from_pandas(df)

## Multi-Tier TTR Model

The `TTRSolver` callable class reconstructs a `DisruptionScenario` from each row and runs it through the exact same (unmodified) `run_scenario_ttr` wrapper used in `06_realistic_stress_testing`, picking the right company's dataset for each row.

In [0]:
class TTRSolver:
    """Callable class to run the Pyomo model for a single disrupted scenario."""

    def __init__(self, datasets=complex_datasets):
        self.datasets = complex_datasets

    def __call__(self, row):
        dataset = self.datasets[row["company"]]
        scenario = ds_lib.DisruptionScenario(
            scenario_id=row["scenario_id"],
            name=row["scenario_name"],
            description="",
            real_world_basis=row["real_world_basis"],
            scenario_type=row["scenario_type"],
            disrupted_nodes=list(row["disrupted_nodes"]),
            ttr=row["ttr"],
        )
        result = ds_lib.run_scenario_ttr(dataset, scenario)
        row["termination_condition"] = str(result.iloc[0]["termination_condition"])
        row["lost_profit"] = result.iloc[0]["lost_profit"]
        return row

### Test the Solver on a Single Row

In [0]:
TTRSolver()(df.take(1)[0])

### Distributed Computation with Ray Data API

As in `03_stress_testing (large network)`, we repartition for parallelism and map the solver across the cluster. Adjust `concurrency` based on your cluster size.

In [0]:
df_ttr = df.repartition(300).map(TTRSolver,
       num_cpus=1,
       concurrency=(4,20))
pandas_df_ttr = df_ttr.to_pandas()

### Highest Risk Scenarios
Top 10 across both companies by lost profit.

In [0]:
highest_risk_scenarios = pandas_df_ttr.sort_values(by="lost_profit", ascending=False)[0:10]
display(highest_risk_scenarios[["company", "scenario_id", "scenario_name", "scenario_type", "ttr", "lost_profit"]])

## Multi-Tier TTS Model

Same pattern, using `run_scenario_tts`. Note that `run_scenario_tts` normalizes any "unbounded" solver termination (the network can absorb the disruption indefinitely, given enough redundancy/headroom) to `tts = inf` rather than trusting the solver's raw — and in that case, meaningless — objective value; see the docstring in `scripts/disruption_scenarios.py` for why.

In [0]:
class TTSSolver:
    """Callable class to run the Pyomo model for a single disrupted scenario."""

    def __init__(self, datasets=complex_datasets):
        self.datasets = complex_datasets

    def __call__(self, row):
        dataset = self.datasets[row["company"]]
        scenario = ds_lib.DisruptionScenario(
            scenario_id=row["scenario_id"],
            name=row["scenario_name"],
            description="",
            real_world_basis=row["real_world_basis"],
            scenario_type=row["scenario_type"],
            disrupted_nodes=list(row["disrupted_nodes"]),
            ttr=row["ttr"],
        )
        result = ds_lib.run_scenario_tts(dataset, scenario)
        row["termination_condition"] = str(result.iloc[0]["termination_condition"])
        row["tts"] = result.iloc[0]["tts"]
        return row

In [0]:
TTSSolver()(df.take(1)[0])

In [0]:
df_tts = df.repartition(300).map(TTSSolver,
                                 num_cpus=1,
                                 concurrency=(4,20))
pandas_df_tts = df_tts.to_pandas()

### Analyze Results

`tts` is `inf` for scenarios the network survives indefinitely, so we exclude those from the histogram (they'd all fall in one bucket) and report their count separately.

In [0]:
import matplotlib.pyplot as plt

merged = pandas_df_ttr.merge(
    pandas_df_tts[["company", "scenario_id", "tts"]], on=["company", "scenario_id"]
)
n_unbounded = (merged["tts"] == float("inf")).sum()
print(f"{n_unbounded} of {len(merged)} scenarios survive indefinitely (tts = inf)")

bounded = merged[merged["tts"] != float("inf")].copy()
bounded["delta"] = bounded["ttr"] - bounded["tts"]
ax = bounded.hist(column="delta", bins=20, grid=False, edgecolor="black", figsize=(10, 6))
plt.title("Histogram of TTR - TTS (bounded scenarios only)")
plt.xlabel("TTR - TTS")
plt.ylabel("Frequency")
plt.grid(axis="y", alpha=0.75)
plt.show()

Nodes with a negative TTR − TTS are not a concern — the network survives longer than the disruption lasts. Nodes with a positive TTR − TTS (and the disruptions the curated real-world events model) are where mitigation investment — supplier-specific inventory buffers, faster recovery agreements, or geographic sourcing diversification — should be prioritized.

## Optional: Planet-Scale Sample Sweep

This standalone section is **not required** for the main pipeline above and does not affect `pandas_df_ttr`/`pandas_df_tts` or the Delta tables written later — it demonstrates running the same `TTRSolver`/`TTSSolver` classes against the much larger `"planet"` scale preset (~76,000 nodes per company) generated in `05_realistic_operational_data`'s optional planet-scale cell.

At this scale an exhaustive one-scenario-per-node sweep means tens of thousands of whole-network LP solves (see the README's "Scaling to planet-scale networks" section for measured solve times), so `single_supplier_failure_scenarios` is called with `sample_fraction=0.15`: every `monopoly_bottleneck`/`oligopoly` node is kept, and only 15% of the remaining nodes are sampled. No new solver logic is introduced — this reuses `TTRSolver`/`TTSSolver` exactly as defined above, just re-parameterizing the dataset and scenario-building step.

Controlled by the `run_planet_scale` widget (default `"yes"`) — set it to `"no"` to skip this section; it's slower than the main pipeline above.

In [ ]:
if run_planet_scale:
    planet_datasets = {}
    for company in ("nvidia", "apple"):
        with open(f"/Volumes/{catalog}/{read_schema}/{volume}/dataset_realistic_{company}_planet.json", "r") as f:
            planet_datasets[company] = json.load(f)

    def build_planet_scenario_rows(company, dataset):
        rng = random.Random(1)
        scenarios = ds_lib.single_supplier_failure_scenarios(
            dataset, rng, ttr_lo=5, ttr_hi=60, sample_fraction=0.15
        )
        scenarios += ds_lib.named_real_world_scenarios(dataset)
        return [
            {
                "company": company,
                "scenario_id": s.scenario_id,
                "scenario_name": s.name,
                "scenario_type": s.scenario_type,
                "real_world_basis": s.real_world_basis,
                "disrupted_nodes": list(s.disrupted_nodes),
                "ttr": s.ttr,
            }
            for s in scenarios
        ]

    planet_rows = []
    for company, dataset in planet_datasets.items():
        planet_rows.extend(build_planet_scenario_rows(company, dataset))

    df_planet = pd.DataFrame(planet_rows)
    print(f"{len(df_planet)} total planet-scale scenarios across {df_planet['company'].nunique()} companies")
    df_planet = ray.data.from_pandas(df_planet)
else:
    print("run_planet_scale is 'no' — skipping planet-scale sweep")

In [ ]:
if run_planet_scale:
    class PlanetTTRSolver(TTRSolver):
        """Same solver logic as TTRSolver, pointed at the planet-scale datasets."""

        def __init__(self, datasets=planet_datasets):
            self.datasets = datasets

    df_planet_ttr = df_planet.repartition(300).map(PlanetTTRSolver, num_cpus=1, concurrency=(4, 20))
    pandas_df_planet_ttr = df_planet_ttr.to_pandas()

    highest_risk_planet_scenarios = pandas_df_planet_ttr.sort_values(by="lost_profit", ascending=False)[0:10]
    display(highest_risk_planet_scenarios[["company", "scenario_id", "scenario_name", "scenario_type", "ttr", "lost_profit"]])

## Shutdown Ray Cluster

In [0]:
try:
    shutdown_ray_cluster()
except Exception:
    pass
try:
    ray.shutdown()
except Exception:
    pass

## Write to Delta Tables

In [0]:
try:
    spark.createDataFrame(pandas_df_ttr.drop(columns=["disrupted_nodes"])).write.mode("overwrite").saveAsTable(
        f"{catalog}.{write_schema}.stress_test_result_ttr_realistic_complex"
    )
except Exception as e:
    print(f"Warning: Could not save to Delta table: {e}")

try:
    spark.createDataFrame(pandas_df_tts.drop(columns=["disrupted_nodes"])).write.mode("overwrite").saveAsTable(
        f"{catalog}.{write_schema}.stress_test_result_tts_realistic_complex"
    )
except Exception as e:
    print(f"Warning: Could not save to Delta table: {e}")

## Wrap Up

We ran the named disruption-scenario library — thousands of single-supplier failures plus real-world-event-inspired regional and material-wide shortages — across two ~2,300-node, real-company-anchored supply chains, using the exact same Ray-distributed pattern as `03_stress_testing (large network)`. This concludes the realistic scenario ladder: simple → medium → complex (Nvidia-like/Apple-like), all built on the same unmodified time-to-recover/time-to-survive optimization engine introduced in `02_stress_testing (small network)`. See the README's "Realistic Stress-Test Scenario Ladder" section for the full methodology, data provenance, and citations.

&copy; 2025 Databricks, Inc. All rights reserved. The source in this notebook is provided subject to the Databricks License [https://databricks.com/db-license-source].  All included or referenced third party libraries are subject to the licenses set forth below.

| library                                | description             | license    | source                                              |
|----------------------------------------|-------------------------|------------|-----------------------------------------------------|
| pyomo | An object-oriented algebraic modeling language in Python for structured optimization problems | BSD-3 | https://pypi.org/project/pyomo/
| highspy | Linear optimization solver (HiGHS) | MIT | https://pypi.org/project/highspy/
| ray | Framework for scaling AI/Python applications | Apache 2.0 | https://github.com/ray-project/ray